<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg" align="left">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.991 · Aprenentatge automàtic · PEC2</p>
<p style="margin: 0; text-align:right;">2025-2 · Màster universitari en Ciència de dades (Data science)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicació</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>


# PEC 2: Aprenentatge no supervisat

Al llarg d'aquesta pràctica veurem com aplicar distintes tècniques no supervisades: clustering de blobs clàssics (*k-means* i la regla del colze), clustering amb formes i feature engineering.

Així com algunes de les seves aplicacions reals: reducció de dimensionalitat, agrupació i generació d'imatges.

**Notes importants:**

 - Cadascun dels exercicis pot suposar diversos minuts d'execució, per la qual cosa el lliurament s'ha de fer en format notebook i en format html, on es vegi el codi, els resultats i comentaris de cada exercici. Es pot exportar el notebook a html des del menú File $\to$ Download as $\to$ HTML.

 - Existeix un tipus de celda especial per a allotjar text. Aquest tipus de celda us serà molt útil per a respondre a les diferents preguntes teòriques plantejades al llarg de cada PEC. Per a canviar el tipus de celda a aquest tipus, trieu en el menú: Cell $\to$ Cell Type $\to$ Markdown.

 - La solució plantejada no ha d'utilitzar mètodes, funcions o paràmetres declarats "deprecated" en futures versions.

 - És convenient que utilitzeu una llavor amb un valor fix (en aquest Notebook se us proposa la variable _seed_ inicialitzada a 100) en tots aquells mètodes o funcions que continguin alguna component aleatòria per a assegurar-vos que obtindreu sempre el mateix resultat en les distintes execucions del vostre codi.

 - No es permet l'ús d'eines d'intel·ligència artificial per a resoldre les qüestions de la PEC.

 - No oblideu posar el vostre nom i cognoms en la següent celda.


<div style="background-color: #ff9999; border-color: #dfb5b4; border-left: 5px solid #dfb5b4; padding: 0.5em;">
<strong>AVÍS:</strong> tingueu cura, més podria ser menys. Si us excediu en les respostes, podríeu tenir penalització en la nota. Sigueu concisos, sintetitzeu el que voleu remarcar i aneu al gra si us plau.
</div>

<div class="alert alert-block alert-info">
<strong>Nom i cognoms:</strong>
</div>

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import mode
import seaborn as sns
from sklearn import cluster, datasets, manifold, decomposition, ensemble
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from umap import UMAP

## 1. Clustering clàssic (4 punts)

### 1.1 Clustering de blobs: *k-means* i la regla del colze

Anem a generar un dataset compost per $n$ núvols de punts, on $n$ serà un número aleatori entre 2 i 4, usant el mòdul ```datasets``` de scikit-learn.

In [ ]:
N_SAMPLES = 1500
MIN_CLUSTERS = 2
MAX_CLUSTERS = 4
X, y = datasets.make_blobs(n_samples=N_SAMPLES,
                           n_features=2,
                           centers=random.randint(MIN_CLUSTERS, MAX_CLUSTERS),
                           center_box=(-20, 20),
                           cluster_std=.6)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.scatter(X[:,0], X[:,1], s=2)
ax.axis('equal')
plt.tight_layout()

Una tècnica per a estimar $k$ és, tal com s'explica en la teoria:
> Els criteris anteriors (minimització de distàncies intra grup o maximització de distàncies inter grup) poden usar-se per a establir un valor adequat per al paràmetre k. Valors k per als quals ja no s'aconsegueixen millores significatives en l'homogeneïtat interna dels segments o l'heterogeneïtat entre segments distints, haurien de descartar-se.

El que popularment es coneix com a *regla del colze*.

Primer és necessari calcular la suma dels errors quadràtics ([*SSE*](https://bl.ocks.org/rpgove/0060ff3b656618e9136b)) que consisteix en la suma de tots els errors (distància de cada punt al seu centroide assignat) al quadrat.

$$SSE = \sum_{i=1}^{K} \sum_{x \in C_i} euclidean(x, c_i)^2$$

On $K$ és el nombre de clusters a buscar per *k-means*, $x \in C_i$ són els punts que pertanyen a l'i-èsim cluster, $c_i$ és el centroide del cluster $C_i$ (al qual pertany el punt $x$), i $euclidean$ és la [distància euclidiana](https://en.wikipedia.org/wiki/Euclidean_distance).

Aquest procediment realitzat per a cada possible valor $k$, resulta en una funció monòtona decreixent, on l'eix $x$ representa els distints valors de $k$, i l'eix $y$ el $SSE$. Intuïtivament es podrà observar un significatiu descens de l'error, que indicarà el valor idoni de $k$.

**Es demana realitzar la representació gràfica de la regla del colze juntament amb la seva interpretació, utilitzant la llibreria
http://googleusercontent.com/immersive_entry_chip/0

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> càlcul i visualització de la regla del colze.  
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Què s'interpreta en la gràfica? Com podria millorar-se l'elecció de $k$?.
</div>

### 1.2 Clustering amb formes i feature engineering

Però no tots els datasets són com els de l'exercici anterior. Per a aquesta segona part anem a emprar el següent conjunt de dades:

In [ ]:
data_circles = ('circles', *datasets.make_circles(n_samples=N_SAMPLES, factor=.5, noise=.05))

On *data_circles* és una tupla amb tres posicions: el nom del dataset i els dos valors retornats per la funció que genera el dataset:

In [ ]:
datasets.make_circles?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(data_circles[1][:,0], data_circles[1][:,1], c=data_circles[2], s=2)
ax.set_title('Dataset {}'.format(data_circles[0]))
plt.tight_layout()

#### 1.2.1 Trobant els clústers amb *k-means*

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> aplica la regla del colze per decidir el valor de $k$.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> aplica <i>k-means</i> amb el valor de $k$ triat.
<br>
Visualitza el resultat en un <i>scatter plot</i> representant cada cluster amb un color distint.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Què ha succeït? Explica els motius pels quals creus que s'ha produït aquest resultat.  
</div>

#### 1.2.2 Més enllà de K-Means: algorismes basats en densitat

En aquest apartat es demana aplicar clustering per densitat com [DBSCAN](https://en.wikipedia.org/wiki/DBSCAN) al dataset anterior per poder trobar els dos clusters inicials.

<br>
<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> prova la implementació de <a href="http://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html">DBSCAN en scikit-learn</a> jugant amb els paràmetres <i>eps</i> i <i>min_samples</i> per trobar les 2 estructures subjacents (i <i>outliers</i>).
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Què ha succeït? Explica els motius pels quals creus que s'ha produït aquest resultat.  
</div>

#### 1.2.3 Més enllà de K-Means: algorismes jeràrquics

En aquest apartat es demana visualitzar mitjançant un [dendrograma](https://en.wikipedia.org/wiki/Dendrogram) la construcció progressiva dels grups mitjançant un algorisme jeràrquic aglomeratiu (estratègia *bottom-up*). Amb això es pretén trobar un mètode gràfic per entendre el comportament de l'algorisme i trobar els dos clusters.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong><br>

Prova la implementació de <a href="https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html">clustering jeràrquic de scipy</a> provant distints <a href="https://en.wikipedia.org/wiki/Hierarchical_clustering#Linkage_criteria">criteris d'enllaç o <i>linkage</i></a> permetent identificar els clusters subjacents (mostrant el seu resultat) i el seu dendrograma.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Interpreta el dendrograma i comenta quin criteri d'enllaç s'ha comportat millor. Per què?
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Què ha succeït? Explica els motius pels quals creus que s'ha produït aquest resultat.  
</div>

### 1.3 *Feature engineering* i agrupament

Alguns dels algorismes anteriors es basen en unes suposicions que no es complien en el dataset.
Moltes vegades en lloc d'optar per algorismes més complexos o que requereixen major còmput, es poden transformar les dades per poder aplicar amb èxit tècniques més simples. Això és un clar exemple de *feature engineering*.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Creus que podrien trobar els dos grups amb k-means? Per què?
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> transforma els punts anteriors del dataset a un nou espai de 2 dimensions:
<ul>
<li>Radi, o distància al punt (0,0)
<li>Angle, respecte al vector (1,0)
</ul>
Perquè totes les dimensions tinguin el mateix pes, a més anem a <a href="http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html">normalitzar-les entre 0 i 1 d'acord amb el seu màxim i mínim</a>.
<br>
Visualitzar els punts del "nou" dataset.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Què creus que succeirà en aplicar els anteriors algorismes en aquest "nou" dataset?
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> aplica cadascun dels algorismes d'agrupament anteriors que no hagin pogut localitzar adequadament els dos clusters originals per tractar de trobar-los en aquest "nou" espai. Ajusta els paràmetres necessaris per facilitar la seva detecció.
<br><br>
Per a cada algorisme, visualitza els clusters trobats en 2 imatges:
<ul>
<li> En el "nou" espai (radi i angle).
<li> En l'espai original (posició x i y), però NO amb les etiquetes (pertinença al cluster) obtingudes en aplicar els algorismes sobre el dataset original, sinó amb les etiquetes obtingudes en realitzar el clustering en el "nou" espai. A veure si així s'aconsegueixen solucionar els problemes inicials.
</ul>
</div>

## 2. Descobrint grups d'imatges (6 punts)

En plantejar un problema de classificació amb un dataset de més de tres atributs (dimensions), no es pot realitzar una visualització clàssica del dataset per entendre les dades. Per això, un dels usos dels mètodes de reducció de dimensionalitat és transformar les dades de més de 4 dimensions a 3 o menys per poder visualitzar-les.

### 2.1 Anàlisi de dades

El [dataset Iris](https://es.wikipedia.org/wiki/Iris_flor_conjunto_de_datos) conté 4 atributs sobre tres tipus de flors.

In [ ]:
iris = datasets.load_iris()
X = iris.data   # np.array con shape (150, 4)
y = iris.target # np.array con shape (150,)

Al excedir les 3 dimensions necessitarem més d'una visualització per entendre les dades.

Per solucionar-ho, una alternativa és usar els [*pair plots*](http://seaborn.pydata.org/generated/seaborn.pairplot.html) que enfronten parells de dimensions per tractar de donar una visió global a partir d'un [DataFrame](https://pandas.pydata.org/pandas-docs/stable/dsintro.html):

In [ ]:
# DataFrame
iris_df = pd.DataFrame(np.hstack([X, y.reshape(-1, 1)]), columns=iris.feature_names + ['species'])
sns.pairplot(iris_df, vars=iris.feature_names, hue="species")

### 2.2 Reducció de dimensionalitat

Com a alternativa a les múltiples gràfiques que generen els gràfics aparellats, es planteja utilitzar una tècnica de reducció de dimensionalitat per passar de 4 dimensions a 2. Noti's que gràfiques com *longitud de pètal* vs *amplada de pètal* mostren certa separabilitat (els 3 tipus de flors estan aparentment separats).

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> realitzar una reducció de dimensionalitat amb PCA per passar de 4 dimensions a 2 i crear una visualització on el color dels punts depengui del tipus de flor a la qual pertany.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> El resultat de la reducció de dimensionalitat manté la separabilitat? PCA ens ajuda a la separabilitat? Per què?
</div>

L'ús de tècniques de reducció de dimensionalitat és de gran utilitat quan aquesta és molt alta. Per exemple, el [dataset Digits](http://scikit-learn.org/stable/auto_examples/classification/plot_digits_classification.html) conté 1797 imatges de números del 0 al 9, de 8 per 8 píxels. Si es pren cada píxel com una dimensió, això es tradueix en el fet que cada mostra té 64 dimensions!

In [ ]:
digits = datasets.load_digits()
X = digits.data   # np.array con shape (1797, 64)
y = digits.target # np.array con shape (1797,)

n_clusters = 10
random_state = 42

Exemple dels 24 primers dígits de 8 per 8 píxels juntament amb la seva etiqueta presents en el dataset:

In [ ]:
fig, ax = plt.subplots(3, 8, figsize=(12, 5))
for i, axis in zip(range(24), ax.reshape(-1)):
    axis.imshow(X[i,:].reshape(8, 8), cmap='gray')
    axis.set_title('#{}'.format(y[i]))
plt.tight_layout()

Amb un número tan elevat de dimensions perd sentit visualitzar el dataset amb un *pair plot* i apareixen altres perills com la [maledicció de la dimensionalitat](https://es.wikipedia.org/wiki/Maldici%C3%B3n_de_la_dimensi%C3%B3n).
Per reduir la seva dimensió i així entendre l'estructura de les dades en alta dimensionalitat existeixen distintes alternatives amb resultats molt distints.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> realitzar una reducció de dimensionalitat amb PCA per passar de 64 dimensions a 2 i crear una visualització on el color dels punts depengui del dígit al qual pertany.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> El resultat de la reducció de dimensionalitat manté la separabilitat? Era d'esperar? Per què?
</div>

Reduir la dimensionalitat té una gran implicació: la pèrdua d'informació. Però en quin grau afecta la dada?

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> Tria una mostra aleatòria o concreta del dataset i visualitza-la. Després redueix la dimensionalitat de tot el dataset a 1, 2, 3... fins a 40 dimensions amb PCA i inverteix el procés tractant de reconstruir les "imatges originals". Amb això fes dues coses: (1) Visualitza la imatge resultant de la reconstrucció de la mostra triada després de la reducció a 1, 2, ..., 5 dimensions. (2) Calcula l'error quadràtic mig (MSE) del dataset reconstruït respecte a l'original i visualitza l'error de reconstrucció en funció del nombre de dimensions al qual ha estat reduït (de 1 a 40 dimensions).
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Com afecta la reducció de dimensionalitat respecte a la informació mantinguda de la imatge? És lineal la relació? Per què?
</div>

L'algorisme [t-SNE](https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding) ideat per [van der Maaten i Hinton](https://lvdmaaten.github.io/tsne/) difereix de PCA en el fet que no tracta de maximitzar la variança explicada. Intuïtivament, t-SNE tracta que el veïnat d'un punt en baixa dimensionalitat sigui la mateixa que l'original. Partint d'una localització aleatòria de cada punt, corregeix la seva posició de forma iterativa tractant de minimitzar la distància als seus veïns originals fins a convergir.

Per a això, t-SNE disposa de diversos [paràmetres](https://distill.pub/2016/misread-tsne/) que poden modificar dràsticament el resultat. Pel que es recomana conèixer el seu funcionament abans d'aplicar la tècnica.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> realitzar una reducció de dimensionalitat amb t-SNE per passar de 64 dimensions a 2 i crear una visualització on el color dels punts depengui del dígit al qual pertany.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> El resultat de la reducció de dimensionalitat manté la separabilitat? Era d'esperar? Per què?
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> Tria una mostra aleatòria o concreta del dataset i visualitza-la. Després redueix la dimensionalitat de tot el dataset a 1, 2, 3... fins a 40 dimensions amb t-SNE i inverteix el procés tractant de reconstruir les "imatges originals". Amb això fes dues coses: (1) Visualitza la imatge resultant de la reconstrucció de la mostra triada després de la reducció a 1, 2, ..., 5 dimensions. (2) Calcula l'error quadràtic mig (MSE) del dataset reconstruït respecte a l'original i visualitza l'error de reconstrucció en funció del nombre de dimensions al qual ha estat reduït (de 1 a 40 dimensions).
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Com afecta la reducció de dimensionalitat respecte a la informació mantinguda de la imatge? És lineal la relació? Per què?
</div>

L'algorisme [UMAP](https://umap-learn.readthedocs.io/en/latest/) és més recent i també permet realitzar reduccions de dimensionalitat de manera no lineal, però en aquest cas projectant d'un espai a l'altre, no resolent un problema d'optimització. Hi ha una comparativa molt interessant entre *t-SNE* i *UMAP* [aquí](https://pair-code.github.io/understanding-umap/).

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> realitzar una reducció de dimensionalitat amb <i>UMAP</i> per passar de 64 dimensions a 2 i crear una visualització on el color dels punts depengui del dígit al qual pertany.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> El resultat de la reducció de dimensionalitat manté la separabilitat? ¿Era d'esperar? Per què?
</div>

Reduir la dimensionalitat té una gran implicació: la pèrdua d'informació. Però en quin grau afecta a la dada?

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> Tria una mostra aleatòria o concreta del dataset i visualitza-la. Després redueix la dimensionalitat de tot el dataset a 1, 2, 3... fins a 40 dimensions amb UMAP i inverteix el procés tractant de reconstruir les "imatges originals". Amb això fes dues coses: (1) Visualitza la imatge resultant de la reconstrucció de la mostra triada després de la reducció a 1, 2, ..., 5 dimensions. (2) Calcula l'error quadràtic mig (MSE) del dataset reconstruït respecte a l'original i visualitza l'error de reconstrucció en funció del nombre de dimensions al qual ha estat reduït (de 1 a 40 dimensions).
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Com afecta la reducció de dimensionalitat respecte a la informació mantinguda de la imatge? És lineal la relació? Per què?
</div>

### 2.3 Agrupament d'imatges

Emprant el mateix dataset executa l'algorisme *k-means* ($k=10$) en quatre escenaris: sobre les dades originals (64D), després d'aplicar una reducció *PCA* a 2D, després d'una reducció *t-SNE* a 2D, i després d'una reducció amb *UMAP* a 2D.

Per saber si la reducció de dimensionalitat contribueix de manera positiva o negativa al clustering posterior avaluaràs l'agrupament emprant la mètrica [ARI (Adjusted Rand Index)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html), ja que en aquest cas comptem amb les etiquetes originals del dataset.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> per al primer cas no redueixis la dimensionalitat del dataset, simplement aplica <i>k-means</i> i mostra l'ARI del resultat.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> redueix la dimensionalitat del dataset amb <i>PCA</i>, aplica <i>k-means</i> i mostra l'ARI del resultat.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> redueix la dimensionalitat del dataset amb <i>t-SNE</i>, aplica <i>k-means</i> i mostra l'ARI del resultat.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> redueix la dimensionalitat del dataset amb <i>UMAP</i>, aplica <i>k-means</i> i mostra l'ARI del resultat.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Com ha afectat la reducció de dimensionalitat al clustering posterior? Per què?
</div>

En aprenentatge no supervisat normalment no es disposa d'etiquetes, per la qual poques vegades se sol recórrer a la matriu de confusió per comprovar els resultats. Però com que en aquest cas sí que es disposa d'elles, un exercici interesant és mirar quins números es confonen amb altres números per la seva grafia. Per a això és possible definir una etiqueta al clúster en base a l'etiqueta de moda entre totes les etiquetes presents en les mostres que pertanyen a aquest clúster (jugarien el paper de "prediccions"). D'altra banda, es disposa de les etiquetes reals, per la qual cosa és possible elaborar la matriu de confusió.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> genera la matriu de confusió comparant els dígits reals amb les etiquetes assignades en base a la moda del clúster. Visualitza els resultats en un mapa de calor per detectar quins parells de números presenten similituds en els seus traços (com l'1 amb el 8 o el 3 amb el 9) que provoquen que l'algorisme els agrupi en un mateix clúster.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> Quins dígits tendeixen a confondre's amb quins altres? Té sentit?
</div>

Atès que *k-means* calcula els centroides dels diferents grups que identifica, podem trobar l'exemple "canònic" de cadascun d'ells.


<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> busca 10 grups amb <i>k-means</i> en les dades brutes (alta dimensionalitat), extreu els centroides trobats i visualitza'ls.
</div>

### 2.4 Interpolació d'imatges

Al representar les imatges com a vectors és possible conèixer el que succeeix en punts intermedis de l'espai (molt similar al que succeeix en els models generatius de difusió).

És possible interpolar imatges tant en alta dimensionalitat com reduint, desplaçant-nos en l'espai de baixa dimensionalitat, i invertint la reducció en cadascun dels passos.

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> tria dos centroides (pot ser a l'atzar), pots projectar a baixa dimensionalitat amb algun dels mètodes vists (serà necessari invertir l'operació), calcula les coordenades de 6 punts intermedis en el camí entre els dos centroides i mostra les imatges reconstruïdes de cadascun dels punts intermedis de la interpolació.
</div>

### 2.5 Detecció d'*outliers*

És possible identificar imatges "anòmales" emprant tècniques de detecció d'anomalies com [isolation forest](https://en.wikipedia.org/wiki/Isolation_forest).

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Implementació:</strong> aplica isolation forest al conjunt original i visualitza les 5 imatges amb major grau d'anomalia, indica també la seva etiqueta i la seva puntuació com a outlier proporcionada per isolation forest.
</div>

<div style="background-color: #EDF7FF; border-color: #7C9DBF; border-left: 5px solid #7C9DBF; padding: 0.5em;">
<strong>Anàlisi:</strong> si haguessis de donar-li un significat al concepte d'anomalia en aquest dataset veient els resultats, quin seria?
</div>